# WM 2026 — Live-Schicht (Phase 3)

**Ziel:** Waehrend des Turniers die Prognose mit echten Resultaten aktualisieren,
sodass sich die Titelwahrscheinlichkeiten mit jedem Spieltag schaerfen.

**Architektur (drei Schritte, taeglich):**
```
1. FETCH    gespielte WM-Spiele von TheSportsDB holen
2. UPDATE   Elo mit diesen Resultaten nachziehen + gespielte Spiele fixieren
3. RESIM    nur die verbleibenden Spiele simulieren -> aktuelle Prognose
```

**Wichtig — Timing:** Die WM startet am **11. Juni 2026**. Jetzt gibt es noch keine
Live-Resultate. Dieses Notebook dient zum **Verstehen und Trockentesten**; die echte
Automatik laeuft ueber `src/live_forecast.py` + GitHub Actions (am Ende erklaert).

Die ganze Kernlogik liegt jetzt im Modul `src/wm_model.py` — Notebook und
Automatik-Skript teilen sich denselben, getesteten Code.

## 1. Modul laden und Vorturnier-Prognose

`build_forecast` macht den kompletten Durchlauf: Daten -> Elo -> Tor-Modell ->
Gruppen -> Simulation. Ohne Live-Resultate ist das Ergebnis die Vorturnier-Prognose
(identisch zu Notebook 03).

In [ ]:
import sys
sys.path.insert(0, "../src")          # Modul aus src/ importierbar machen
import wm_model as wm

pre, n = wm.build_forecast(local="../data/results.csv", live_results=None, n_sims=10000,
                           fit_cache="../output/goal_coefs.json")
print(f"Beruecksichtigte Live-Spiele: {n}")
pre.head(8).style.format({"Titel": "{:.1%}", "Finale": "{:.1%}", "Halbfinale": "{:.1%}"})

## 2. Wie Live-Resultate einfliessen

Ein gespieltes Spiel wird als Dict uebergeben (gleiche Felder wie der Datensatz).
`build_forecast` macht damit zweierlei:

- **Elo nachziehen:** Das Resultat wird ans Ende der Spielhistorie gehaengt, die
  Elo-Wertungen der beteiligten Teams aktualisieren sich (echte Update-Formel).
- **Spiel fixieren:** In der Simulation wird genau diese Paarung nicht mehr
  gewuerfelt, sondern mit dem echten Resultat gewertet (Punkte, Tordifferenz,
  K.-o.-Sieger).

Zum Testen injizieren wir ein paar **erfundene** Resultate und schauen, ob die
Wahrscheinlichkeiten plausibel reagieren.

In [ ]:
# Beispiel-Resultate (frei erfunden, nur zum Testen der Mechanik)
test_results = [
    dict(date="2026-06-13", home_team="Brazil", away_team="Morocco",
         home_score=4, away_score=0, neutral=True, tournament="FIFA World Cup"),
    dict(date="2026-06-13", home_team="Qatar", away_team="Switzerland",
         home_score=0, away_score=3, neutral=True, tournament="FIFA World Cup"),
]

after, n = wm.build_forecast(local="../data/results.csv",
                             live_results=test_results, n_sims=10000,
                             fit_cache="../output/goal_coefs.json")
print(f"Beruecksichtigte Live-Spiele: {n}")

# Vorher/Nachher fuer die betroffenen Teams
for team in ["Brazil", "Switzerland", "Morocco", "Qatar"]:
    a = pre[pre.Team == team].Titel.values[0]
    b = after[after.Team == team].Titel.values[0]
    print(f"{team:14s} {a:5.1%} -> {b:5.1%}")

Brasilien und die Schweiz steigen nach ihren Siegen, Marokko und Katar fallen —
die Mechanik greift. (Die Zahlen hier sind nur Demo, weil die Resultate erfunden
sind.)

## 3. Resultate von TheSportsDB holen

Das Automatik-Skript `src/live_forecast.py` enthaelt die Fetch-Logik. Sie sucht die
WM-Liga ueber `all_leagues.php` und holt dann alle Saison-Spiele ueber
`eventsseason.php`; gespielte Partien (mit Resultat) werden zu Live-Daten.

**Drei Punkte, auf die du achten musst:**

- **API-Key:** Setze ihn als Umgebungsvariable `SPORTSDB_KEY` (Default `"123"` =
  kostenloser Test-Key). Niemals fest im Code.
- **Rate-Limit:** Free-Tier = 30 Anfragen/Min. Wir machen nur 2 Anfragen pro Lauf,
  also unkritisch — bei einmal taeglich erst recht.
- **Teamnamen:** TheSportsDB nutzt teils andere Namen als der Trainingsdatensatz
  (z.B. "USA" vs. "United States"). Die `NAME_MAP` im Skript muss beim ersten echten
  Lauf gegen die tatsaechlichen API-Namen geprueft werden.

Hier rufen wir die Fetch-Funktion testweise auf — vor Turnierstart liefert sie eine
leere Liste (es gibt noch nichts zu holen):

In [3]:
import live_forecast as lf

live = lf.fetch_live_results()
print(f"Aktuell von TheSportsDB geholte gespielte WM-Spiele: {len(live)}")
# Vor dem 11.6. erwartet leer; waehrend des Turniers eine Liste von Spiel-Dicts.

[WARN] Live-Abruf fehlgeschlagen (Expecting value: line 1 column 1 (char 0)) -> Vorturnier-Prognose.
Aktuell von TheSportsDB geholte gespielte WM-Spiele: 0


## 4. Automatik mit GitHub Actions

Du musst das **nicht** taeglich manuell laufen lassen. Der Workflow
`.github/workflows/forecast.yml` erledigt das:

- **Zeitplan:** taeglich 06:00 UTC (= 08:00 Schweizer Sommerzeit), nachdem die
  US-Anpfiffe des Vortags durch sind. Zusaetzlich per Knopf manuell ausloesbar.
- **Ablauf:** Repo auschecken -> Python + Abhaengigkeiten -> `live_forecast.py`
  ausfuehren -> das Ergebnis (`output/forecast.md` und `.json`) zurueck ins Repo
  committen. So hast du jeden Morgen die aktuelle Prognose direkt im Repository.

**Einmalige Einrichtung:**
1. API-Key als Secret hinterlegen: Repo -> **Settings -> Secrets and variables ->
   Actions -> New repository secret**, Name `SPORTSDB_KEY`, Wert = dein Key.
2. Workflow-Datei committen und pushen (liegt schon im Projekt).
3. Unter **Actions** den Workflow einmal manuell starten ("Run workflow"), um zu
   pruefen, dass alles laeuft.

> **Warum ein Skript statt dieses Notebooks?** Notebooks sind zum interaktiven
> Arbeiten gedacht; geplante Laeufe automatisiert man mit einem schlanken
> `.py`-Skript. Beide nutzen aber dieselbe Logik aus `wm_model.py` — kein doppelter
> Code.

## Zusammenfassung & Ausblick

Die Live-Schicht steht und ist getestet: Resultate fliessen in Elo *und* Tabelle
ein, die Restsimulation aktualisiert die Prognose, und die Automatik laeuft ohne
manuelles Zutun.

Ab dem 11. Juni musst du nur sicherstellen, dass das Secret gesetzt und der Workflow
aktiv ist — danach erscheint jeden Morgen eine frische `output/forecast.md` im Repo.

**Moegliche Verfeinerungen danach:**
- Teamnamen-Mapping gegen die echten TheSportsDB-Namen abgleichen (erster Lauf).
- Offizielles FIFA-Bracket statt Finish-Seeding (Phase 2b).
- Prognose-Verlauf ueber die Zeit speichern und visualisieren (wie sich die Chancen
  je Team entwickeln).